In [13]:
# Идея 1. Наша затея требует большого количества передвижений по городу (и за городом)
# С задачей передвижений по Москве неплохо справляются различные сервисы.
# Для удобных перемещений по Подмосковью воспользуемся API яндекс расписаний.
# Будем показывать список ближайших электричек в зависимости от направления.

import requests
from datetime import datetime

API_KEY = ''
BASE_URL = "https://api.rasp.yandex-net.ru/v3.0/search/"

station_codes = {
    "Москва-Ярославская": "s2000002",
    "Сергиев Посад": "s9601389",
    "Москва-Белорусская": "s2000006",
    "Звенигород": "s9601368",
    "Тушинская": "s9600741",
    "Новоиерусалимская": "s9601222",
    "Ленинградский вокзал": "s2006004",
    "Подсолнечная": "s9603468",
    "Выхино": "s9601627",
    "Коломна": "s9601311",
    "Царицыно": "s9600891",
    "Серпухов": "s9600830"
}

# коды станций найдены из адресной строки по инструкции

def get_suburban_trains(from_code, to_code, date_str):
    url = BASE_URL
    params = {
        "apikey": API_KEY,
        "from": from_code,
        "to": to_code,
        "date": date_str,
        "transport_types": "suburban",
        "lang": "ru_RU",
        "limit" : 500
    }
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        return []
    data = resp.json()
    trains = []
    for seg in data.get("segments", []):
        departure = seg.get("departure", "").replace('T', ' ').replace('+03:00', '')
        arrival = seg.get("arrival", "").replace('T', ' ').replace('+03:00', '')
        trains.append({
            "number": seg.get("thread", {}).get("number"),
            "departure": departure,
            "arrival": arrival,
            "duration": seg.get("duration")
        })
    return trains

# Маршруты: номер -> (откуда, куда)
routes = {
    1: ("Москва-Ярославская", "Сергиев Посад"),
    2: ("Москва-Белорусская", "Звенигород"),
    3: ("Тушинская", "Новоиерусалимская"),
    4: ("Ленинградский вокзал", "Подсолнечная"),
    5: ("Выхино", "Коломна"),
    6: ("Царицыно", "Серпухов")
}

print("Выберите маршрут:")
print("1 - Москва-Ярославская → Сергиев Посад")
print("2 - Москва-Белорусская → Звенигород")
print("3 - Тушинская → Новоиерусалимская")
print("4 - Ленинградский вокзал → Подсолнечная")
print("5 - Выхино → Коломна")
print("6 - Царицыно → Серпухов")

choice = input("Введите цифру (1-6): ").strip()
if not choice.isdigit() or int(choice) not in routes:
    print("Неверный ввод. Запустите программу снова.")
else:
    print("Введите примерный час отправления в 24-часовом формате")
    hour = int(input())
    while not (hour >= 0 and hour < 24):
      print('Введите корректное время')
      hour = int(input())
    num = int(choice)
    from_name, to_name = routes[num]
    from_code = station_codes[from_name]
    to_code = station_codes[to_name]
    today = datetime.now().strftime("%Y-%m-%d")

    print(f"\nМаршрут: {from_name} → {to_name}")
    print(f"Дата: {today}\n")

    trains = get_suburban_trains(from_code, to_code, today)
    if not trains:
        print("Электричек не найдено.")
    else:
        print(f"Рейсы до станции {routes[int(choice)][1]} в желаемое время:\n")
        cnt = 1
        for t in trains:
            dur_sec = t['duration']
            if dur_sec:
                minutes = dur_sec // 60
                h = minutes // 60
                m = minutes % 60
                dur_str = f"{int(h)} ч {int(m)} мин в пути"
            h_dep = t['departure'].split()[1].split(':')[0].lstrip('0')
            h_dep = int(h_dep) if h_dep else 0
            if hour >= h_dep - 1 and hour <= h_dep + 1:
                print(f"{cnt}. №{t['number']}   {t['departure'].split()[1]} → {t['arrival'].split()[1]}   ({dur_str})")
                cnt += 1

Выберите маршрут:
1 - Москва-Ярославская → Сергиев Посад
2 - Москва-Белорусская → Звенигород
3 - Тушинская → Новоиерусалимская
4 - Ленинградский вокзал → Подсолнечная
5 - Выхино → Коломна
6 - Царицыно → Серпухов
Введите цифру (1-6): 5
Введите примерный час отправления в 24-часовом формате
18

Маршрут: Выхино → Коломна
Дата: 2026-05-24

Рейсы до станции Коломна в желаемое время:

1. №6130   17:40:00 → 19:37:00   (1 ч 57 мин в пути)
2. №6132   18:20:00 → 20:08:00   (1 ч 48 мин в пути)
3. №7186   19:06:00 → 20:37:00   (1 ч 31 мин в пути)
4. №6134   19:20:00 → 21:05:00   (1 ч 45 мин в пути)
5. №6996   19:40:00 → 21:15:00   (1 ч 35 мин в пути)
6. №6136   19:50:00 → 21:39:00   (1 ч 49 мин в пути)
